# Bezpečnost databází

**SQL injection** je technika útoku, při které útočník vloží do vstupu části SQL kódu a ovlivní tak chování dotazu. Řadí se do **OWASP Top 10** nejkritičtějších bezpečnostních rizik webových aplikací.

---

## Přehled pojmů

| Pojem | Popis |
|---|---|
| **SQL injection** | Vložení SQL kódu do vstupu aplikace za účelem manipulace s databází |
| **Parametrizovaný dotaz** | Dotaz, kde jsou hodnoty odděleny od SQL kódu |
| **Hashování hesla** | Jednosměrná transformace hesla — v DB se nikdy neukládá plaintext |
| **Least privilege** | Uživatel DB má jen ta oprávnění, která opravdu potřebuje |
| **Validace vstupu** | Ověření, že vstup odpovídá očekávané formě před použitím v dotazu |

## Proč je to důležité?

| Riziko | Dopad |
|---|---|
| SQL injection | Únik dat, smazání tabulek, obejití přihlášení |
| Plaintext hesla | Při úniku DB jsou okamžitě odhalena všechna hesla |
| Příliš silná práva | Útočník získá přístup i tam, kde nemá |
| Chybějící validace | Aplikace přijme škodlivá nebo nesmyslná data |

In [ ]:
! pip install mysql-connector-python werkzeug

## Připojení a příprava tabulek

In [ ]:
import mysql.connector
from werkzeug.security import generate_password_hash, check_password_hash
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

mycursor.execute("DROP TABLE IF EXISTS prihlaseni_log")
mycursor.execute("DROP TABLE IF EXISTS uzivatele")

mycursor.execute("""
    CREATE TABLE uzivatele (
        id INT PRIMARY KEY AUTO_INCREMENT,
        jmeno VARCHAR(50) NOT NULL UNIQUE,
        heslo_hash VARCHAR(255) NOT NULL,
        email VARCHAR(100) UNIQUE,
        role VARCHAR(20) DEFAULT 'user'
    )
""")

mycursor.execute("""
    CREATE TABLE prihlaseni_log (
        log_id INT PRIMARY KEY AUTO_INCREMENT,
        jmeno VARCHAR(50),
        uspesne TINYINT(1),
        cas TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")

hash_admin = generate_password_hash('tajneheslo123')
mycursor.execute(
    'INSERT INTO uzivatele (jmeno, heslo_hash, email, role) VALUES (%s, %s, %s, %s)',
    ('admin', hash_admin, 'admin@skola.cz', 'admin')
)
mydb.commit()
print('✅ Tabulky připraveny.')

---

## SQL Injection — co to je?

Klasický útok: útočník vloží do vstupního pole (login, vyhledávání) speciální znaky tak, aby pozměnil strukturu SQL dotazu.

### Příklad zranitelného kódu (Python)

```python
# ❌ ŠPATNĚ — uživatelský vstup přímo v SQL dotazu!
def zranitelne_prihlaseni(jmeno):
    sql = "SELECT * FROM uzivatele WHERE jmeno = '" + jmeno + "'"
    mycursor.execute(sql)  # NEBEZPEČNÉ!
    return mycursor.fetchone()
```

### Ukázka útoku

| Vstup útočníka | Výsledný SQL dotaz | Efekt |
|---|---|---|
| `admin` | `SELECT ... WHERE jmeno = 'admin'` | Normální dotaz |
| `' OR '1'='1` | `SELECT ... WHERE jmeno = '' OR '1'='1'` | Vrátí VŠECHNY záznamy |
| `'; DROP TABLE uzivatele; --` | `SELECT ...; DROP TABLE uzivatele; --` | Pokus o smazání tabulky |

> ⚠️ **Pravidlo:** Nikdy neskládejte SQL dotaz pomocí zřetězení s uživatelskými daty!

In [ ]:
# ❌ ZRANITELNÁ funkce — NIKDY takto nepoužívat!
def zranitelne_prihlaseni(jmeno):
    q = chr(39)  # apostrof
    sql = 'SELECT id, jmeno, role FROM uzivatele WHERE jmeno = ' + q + jmeno + q
    print('  SQL: ' + sql)
    mycursor.execute(sql)
    return mycursor.fetchone()

print('=== Ukázka SQL injection ===')
print()

print('1) Normální přihlášení:')
vysledek = zranitelne_prihlaseni('admin')
print('   Výsledek: ' + str(vysledek))
print()

# Injection: ' OR '1'='1
injekce = chr(39) + ' OR ' + chr(39) + '1' + chr(39) + '=' + chr(39) + '1'
print('2) SQL injection — obejití přihlášení:')
print('   Vstup útočníka: ' + injekce)
vysledek = zranitelne_prihlaseni(injekce)
print('   Výsledek: ' + str(vysledek))
print()

print('3) SQL injection — pokus o DROP TABLE:')
try:
    drop_inj = chr(39) + '; DROP TABLE uzivatele; --'
    zranitelne_prihlaseni(drop_inj)
except Exception as e:
    print('   ❌ Driver zachytil chybu: ' + str(e))

---

## Parametrizované dotazy — základní obrana

Při parametrizovaném dotazu se hodnoty **nepředávají jako součást SQL textu**, ale jako oddělené parametry. Databáze je zpracuje jako data, nikoliv jako kód.

### Syntaxe v Pythonu

| Knihovna | Zástupný symbol | Příklad |
|---|---|---|
| `mysql-connector` | `%s` | `cursor.execute('SELECT ... WHERE id = %s', (id,))` |
| `sqlite3` | `?` | `cursor.execute('SELECT ... WHERE id = ?', (id,))` |

```python
# ✅ SPRÁVNĚ — parametrizovaný dotaz
mycursor.execute(
    'SELECT * FROM uzivatele WHERE jmeno = %s',
    (jmeno,)   # parametry jako tuple — čárka za jedinou hodnotou je povinná!
)
```

> 💡 **Pozor na tuple:** I při jednom parametru musí být tuple: `(hodnota,)` — čárka je povinná!

In [ ]:
# ✅ BEZPEČNÁ funkce s parametrizovaným dotazem
def bezpecne_prihlaseni(jmeno, heslo):
    mycursor.execute(
        'SELECT id, jmeno, heslo_hash, role FROM uzivatele WHERE jmeno = %s',
        (jmeno,)
    )
    uzivatel = mycursor.fetchone()
    if uzivatel and check_password_hash(uzivatel[2], heslo):
        return uzivatel
    return None

print('=== Test bezpečné funkce ===')
print()

print('1) Správné přihlášení:')
vy = bezpecne_prihlaseni('admin', 'tajneheslo123')
print('   ' + ('✅ Přihlášen: ' + str(vy[1]) if vy else '❌ Zamítnuto'))
print()

print('2) Špatné heslo:')
vy = bezpecne_prihlaseni('admin', 'spatneheslo')
print('   ' + ('✅ Přihlášen' if vy else '❌ Zamítnuto'))
print()

print('3) SQL injection pokus (stejný vstup jako dříve):')
injekce = chr(39) + ' OR ' + chr(39) + '1' + chr(39) + '=' + chr(39) + '1'
vy = bezpecne_prihlaseni(injekce, 'cokoliv')
print('   ' + ('✅ Přihlášen' if vy else '❌ Zamítnuto — injection selhal'))

---

## Hashování hesel

Hesla se **nikdy** neukládají v otevřené podobě (plaintext). Místo toho se ukládá **jednosměrný hash** — z hashe nelze zpětně získat původní heslo.

### Werkzeug funkce

| Funkce | Popis |
|---|---|
| `generate_password_hash(heslo)` | Vygeneruje bezpečný hash (s automatickým solením) |
| `check_password_hash(hash, heslo)` | Ověří, zda heslo odpovídá uloženému hashi |

### Vlastnosti hashování

- Každé volání `generate_password_hash` vrátí **jiný hash** (salt — náhodná přísada — se liší)
- Přesto `check_password_hash` správně ověří heslo
- Pokud dojde k úniku DB, útočník vidí pouze hash — prolomení trvá velmi dlouho

> ⚠️ **Nikdy** nepoužívejte prosté MD5 nebo SHA-1 bez soli — pro ukládání hesel jsou **nedostatečné**!

In [ ]:
from werkzeug.security import generate_password_hash, check_password_hash

heslo = 'MojeHeslo123'

hash1 = generate_password_hash(heslo)
hash2 = generate_password_hash(heslo)  # pokaždé jiný hash (jiná salt)

print('Heslo:  ' + heslo)
print('Hash 1: ' + hash1)
print('Hash 2: ' + hash2)
print('Hash 1 == Hash 2: ' + str(hash1 == hash2) + '  ← každý hash je unikátní (salt)')
print()
print('check_password_hash(hash1, heslo):    ' + str(check_password_hash(hash1, heslo)))
print('check_password_hash(hash1, spatne):   ' + str(check_password_hash(hash1, 'spatne')))

---

## Bezpečná registrace uživatele

Kombinuje všechny dosud probrané techniky:
- **Validace vstupu** — ověření délky hesla před zpracováním
- **Hashování hesla** — nikdy neuložit plaintext
- **Parametrizovaný INSERT** — ochrana před SQL injection
- **Ošetření výjimek** — `IntegrityError` při duplicitním jménu nebo e-mailu

In [ ]:
def registrace(jmeno, heslo, email):
    if len(heslo) < 8:
        print('❌ Heslo musí mít alespoň 8 znaků.')
        return False
    hash_hesla = generate_password_hash(heslo)
    try:
        mycursor.execute(
            'INSERT INTO uzivatele (jmeno, heslo_hash, email) VALUES (%s, %s, %s)',
            (jmeno, hash_hesla, email)
        )
        mydb.commit()
        print('✅ Uživatel ' + jmeno + ' registrován.')
        return True
    except mysql.connector.errors.IntegrityError:
        mydb.rollback()
        print('❌ Uživatel ' + jmeno + ' nebo e-mail ' + email + ' již existuje.')
        return False

registrace('jan.novak', 'bezpecne123', 'jan@skola.cz')
registrace('jan.novak', 'jinaheslo1', 'jiny@skola.cz')   # duplicitní jméno
registrace('petra', 'kratke', 'petra@skola.cz')            # příliš krátké heslo

mycursor.execute('SELECT id, jmeno, LEFT(heslo_hash, 40), email, role FROM uzivatele')
print()
print('📋 Uživatelé v DB:')
for row in mycursor.fetchall():
    print('  ' + str(row))

---

## Logování přihlášení

Logování úspěšných i neúspěšných přihlášení umožňuje:
- Detekovat **brute-force** útoky (opakující se neúspěšné pokusy)
- Auditovat přístup k systému
- Ladit chyby v autentizaci

In [ ]:
def prihlaseni_s_logem(jmeno, heslo):
    mycursor.execute(
        'SELECT id, jmeno, heslo_hash FROM uzivatele WHERE jmeno = %s',
        (jmeno,)
    )
    uzivatel = mycursor.fetchone()
    uspesne = 1 if uzivatel and check_password_hash(uzivatel[2], heslo) else 0

    # Zalogování výsledku bez ohledu na úspěch
    mycursor.execute(
        'INSERT INTO prihlaseni_log (jmeno, uspesne) VALUES (%s, %s)',
        (jmeno, uspesne)
    )
    mydb.commit()
    return uspesne == 1

print(prihlaseni_s_logem('admin', 'tajneheslo123'))   # True
print(prihlaseni_s_logem('admin', 'spatneheslo'))      # False
print(prihlaseni_s_logem('neexistuje', 'heslo'))       # False

mycursor.execute('SELECT log_id, jmeno, uspesne, cas FROM prihlaseni_log')
print()
print('📋 Log přihlášení:')
print('{:<8} {:<15} {:<10} cas'.format('log_id', 'jmeno', 'uspesne'))
print('-' * 50)
for row in mycursor.fetchall():
    print('{:<8} {:<15} {:<10} {}'.format(row[0], str(row[1]), str(row[2]), row[3]))

---

## Privilegia uživatelů — Least Privilege

Databázový uživatel by měl mít **pouze ta oprávnění, která opravdu potřebuje**.

| Typ uživatele | Vhodná oprávnění |
|---|---|
| Webová aplikace (čtení) | `SELECT` |
| Webová aplikace (zápis) | `SELECT`, `INSERT`, `UPDATE` |
| Administrátor aplikace | `SELECT`, `INSERT`, `UPDATE`, `DELETE` |
| Správce DB | `CREATE`, `DROP`, `GRANT`, ... |

### SQL příkazy pro správu oprávnění

```sql
-- Vytvoření uživatele pouze pro čtení:
CREATE USER 'cteni_user'@'localhost' IDENTIFIED BY 'heslo';
GRANT SELECT ON databaze.* TO 'cteni_user'@'localhost';

-- Uživatel pro aplikaci (bez DROP a DELETE):
CREATE USER 'app_user'@'localhost' IDENTIFIED BY 'heslo';
GRANT SELECT, INSERT, UPDATE ON databaze.uzivatele TO 'app_user'@'localhost';

-- Odebrání oprávnění:
REVOKE INSERT ON databaze.uzivatele FROM 'app_user'@'localhost';

-- Smazání uživatele:
DROP USER 'cteni_user'@'localhost';
```

> 💡 Tyto příkazy musí spustit uživatel s právem `GRANT OPTION` (typicky `root`).

In [ ]:
# Zobrazení práv aktuálního uživatele
try:
    mycursor.execute('SHOW GRANTS FOR CURRENT_USER()')
    print('🔑 Práva aktuálního uživatele:')
    for row in mycursor.fetchall():
        print('  ' + str(row[0]))
except Exception as e:
    print('Chyba: ' + str(e))

---

## Validace vstupu

I při použití parametrizovaných dotazů je vhodné vstup **validovat** — ověřit, že odpovídá očekávanému formátu.

### Doporučené kontroly

| Kontrola | Příklad |
|---|---|
| Délka řetězce | `3 <= len(jmeno) <= 50` |
| Povolené znaky | regulární výraz nebo whitelist |
| Rozsah čísla | `0 <= vek <= 120` |
| Formát e-mailu | obsahuje `@` a `.` za `@` |
| Neprázdná hodnota | `obsah.strip() != ''` |

> 💡 Parametrizovaný dotaz chrání SQL vrstvu, validace chrání **business logiku** — obojí je potřeba!

In [ ]:
import re

def validuj_jmeno(jmeno):
    if not (3 <= len(jmeno) <= 50):
        return False, 'Jméno musí mít 3–50 znaků.'
    if not re.match(r'^[a-zA-Z0-9._-]+$', jmeno):
        return False, 'Jméno obsahuje nepovolené znaky.'
    return True, 'OK'

def validuj_email(email):
    casti = email.split('@')
    if len(casti) != 2 or '.' not in casti[1]:
        return False, 'Neplatný formát e-mailu.'
    return True, 'OK'

testy = [
    ('jan.novak', 'jan@skola.cz'),
    ('ad', 'neplaty-email'),
    ('x' * 60, 'test@test.com'),
    ('platne_jmeno123', 'test@test.com'),
]

print('=== Validace vstupů ===')
print()
for jmeno, email in testy:
    ok_j, msg_j = validuj_jmeno(jmeno)
    ok_e, msg_e = validuj_email(email)
    print('Jméno: ' + repr(jmeno[:30]) + ' => ' + ('✅ OK' if ok_j else '❌ ' + msg_j))
    print('Email: ' + repr(email) + ' => ' + ('✅ OK' if ok_e else '❌ ' + msg_e))
    print()

## Cvičení 1 — Bezpečné ukládání poznámek

1. Vytvořte tabulku `poznamky` (id INT PK AUTO_INCREMENT, uzivatel_id INT FK → uzivatele.id, obsah TEXT NOT NULL, vytvoreno TIMESTAMP DEFAULT CURRENT_TIMESTAMP).
2. Napište funkci `pridej_poznamku(uzivatel_id, obsah)`, která:
   - Ověří, že `obsah` není prázdný
   - Uloží poznámku pomocí **parametrizovaného dotazu**
3. Vložte 2 platné a 1 prázdnou poznámku pro uživatele `id = 1` (admin). Vypište obsah tabulky.

<details>
<summary>🔎 Ukázka řešení (klikni pro zobrazení)</summary>

```python
mycursor.execute('DROP TABLE IF EXISTS poznamky')
mycursor.execute("""
    CREATE TABLE poznamky (
        id INT PRIMARY KEY AUTO_INCREMENT,
        uzivatel_id INT,
        obsah TEXT NOT NULL,
        vytvoreno TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (uzivatel_id) REFERENCES uzivatele(id)
    )
""")

def pridej_poznamku(uzivatel_id, obsah):
    if not obsah or not obsah.strip():
        print('❌ Obsah nesmí být prázdný.')
        return False
    mycursor.execute(
        'INSERT INTO poznamky (uzivatel_id, obsah) VALUES (%s, %s)',
        (uzivatel_id, obsah)
    )
    mydb.commit()
    print('✅ Poznámka uložena.')
    return True

pridej_poznamku(1, 'Nezapomenout aktualizovat server.')
pridej_poznamku(1, 'Zálohovat databázi každý pátek.')
pridej_poznamku(1, '')  # prázdná — odmítnuta

mycursor.execute('SELECT id, uzivatel_id, obsah, vytvoreno FROM poznamky')
for row in mycursor.fetchall():
    print(row)
```

</details>

In [ ]:
# Cvičení 1: Váš kód sem 👇


## Cvičení 2 — Detekce brute-force útoku

1. Napište funkci `pocet_neuspesnych(jmeno, poslednich_minut)`, která pomocí **parametrizovaného dotazu** vrátí počet neúspěšných pokusů o přihlášení za posledních N minut (využijte `NOW()` a `DATE_SUB()`).
2. Napište funkci `je_zamcen(jmeno)`, která vrátí `True`, pokud bylo v posledních 5 minutách více než 3 neúspěšná přihlášení.
3. Otestujte: zalogujte 4 neúspěšná přihlášení pro `'admin'` a ověřte, že `je_zamcen('admin')` vrátí `True`.
4. Proveďte úklid (DROP tabulek) a odpojte se.

<details>
<summary>🔎 Ukázka řešení (klikni pro zobrazení)</summary>

```python
def pocet_neuspesnych(jmeno, poslednich_minut):
    mycursor.execute(
        'SELECT COUNT(*) FROM prihlaseni_log WHERE jmeno = %s AND uspesne = 0 AND cas >= DATE_SUB(NOW(), INTERVAL %s MINUTE)',
        (jmeno, poslednich_minut)
    )
    return mycursor.fetchone()[0]

def je_zamcen(jmeno):
    return pocet_neuspesnych(jmeno, 5) > 3

for _ in range(4):
    mycursor.execute(
        'INSERT INTO prihlaseni_log (jmeno, uspesne) VALUES (%s, %s)',
        ('admin', 0)
    )
mydb.commit()

print('Počet neúspěšných (5 min): ' + str(pocet_neuspesnych('admin', 5)))
print('Je účet zamčen? ' + str(je_zamcen('admin')))
```

</details>

In [ ]:
# Cvičení 2: Váš kód sem 👇


# Úklid na konci:
# mycursor.execute('DROP TABLE IF EXISTS poznamky')
# mycursor.execute('DROP TABLE IF EXISTS prihlaseni_log')
# mycursor.execute('DROP TABLE IF EXISTS uzivatele')
# mydb.commit()
# mydb.close()


---

## 📋 Shrnutí

| Technika | Proč důležitá | Jak v Pythonu |
|---|---|---|
| **Parametrizované dotazy** | Zabrání SQL injection | `cursor.execute(sql, (param,))` |
| **Hashování hesel** | Chrání hesla při úniku DB | `generate_password_hash()` / `check_password_hash()` |
| **Validace vstupu** | Odmítne škodlivá nebo nesmyslná data | Podmínky, `len()`, `re.match()` |
| **Least privilege** | Omezí dopad při průniku | `GRANT SELECT ON ...` |
| **Logování přihlášení** | Detekce útoků, audit přístupu | Tabulka s `TIMESTAMP`, `uspesne` |

### Bezpečnostní checklist

- ☑️ Nikdy neskládejte SQL pomocí zřetězení řetězců s uživatelskými daty
- ☑️ Vždy hashujte hesla (`werkzeug`, `bcrypt`) — nikdy neukládejte plaintext
- ☑️ Validujte vstup před uložením (délka, formát, povolené znaky)
- ☑️ Databázový uživatel má jen minimální potřebná oprávnění
- ☑️ Logujte přihlašování (úspěchy i neúspěchy) — detekce brute-force útoku
- ☑️ Nikdy necommitujte hesla, klíče ani connection stringy do Git repozitáře